In [ ]:
import requests
import json
import pandas as pd
import csv
import re

# Membaca file madaniyah.csv untuk mendapatkan nomor surah
madaniyah_chapters = []
with open('madaniyah.csv', 'r') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)  # Melewati header
    for row in csv_reader:
        madaniyah_chapters.append(int(row[1]))

# Dictionary untuk menyimpan semua DataFrame
dfs = {}

# Melakukan request untuk setiap chapter di madaniyah.csv
for chapter in madaniyah_chapters:
    # URL untuk chapter yang sedang diproses
    url = f"https://api.qurancdn.com/api/v4/verses/by_chapter/{chapter}?language=id&translations=33,134&words=0&fields=text_uthmani&per_page=300"
    
    # Melakukan request ke API
    response = requests.get(url)
    
    # Memeriksa apakah request berhasil
    if response.status_code == 200:
        # Parse JSON response
        data = response.json()
        
        # Membuat list untuk menyimpan data
        arab_texts = []
        king_fahd_texts = []
        kemenag_texts = []
        verse_number = []
        
        # Mengekstrak data yang dibutuhkan
        for verse in data['verses']:
            # Mengambil teks Arab
            arab_texts.append(verse['text_uthmani'])
            
            # Mengambil verse key (nomor ayat)
            verse_number.append(verse['verse_number'])
            
            # Mengambil terjemahan
            for translation in verse['translations']:
                if translation['resource_id'] == 134:
                    # Membersihkan superscript footnote dari teks King Fahd
                    clean_text = re.sub(r'<sup foot_note=\d+>\d+</sup>', '', translation['text'])
                    king_fahd_texts.append(clean_text)
                elif translation['resource_id'] == 33:
                    # Membersihkan superscript footnote dari teks Kemenag
                    clean_text = re.sub(r'<sup foot_note=\d+>\d+</sup>', '', translation['text'])
                    kemenag_texts.append(clean_text)
        
        # Membuat DataFrame
        df = pd.DataFrame({
            'verse_number': verse_number,
            'arab': arab_texts,
            'king_fahd': king_fahd_texts,
            'kemenag': kemenag_texts
        })
        
        # Menyimpan DataFrame ke dalam dictionary dengan nama df_chapter_{chapter}
        var_name = f"df_chapter_{chapter}"
        dfs[var_name] = df
        
        # Membuat variabel global untuk setiap DataFrame
        globals()[var_name] = df
        
        # Menampilkan hasil
        print(f"Berhasil mengambil {len(df)} ayat dari Surah nomor {chapter} dan disimpan ke {var_name}")


In [ ]:
# Import library yang dibutuhkan
import translators as ts
import os
import time
from tqdm import tqdm

# Mendapatkan semua variabel df_chapter
chapter_dfs = [var for var in globals() if var.startswith('df_chapter_')]

print(f"Ditemukan {len(chapter_dfs)} DataFrame untuk ditranslasikan")

# Melakukan translasi untuk setiap DataFrame
for df_name in chapter_dfs:
    # Mendapatkan nomor chapter dari nama variabel
    chapter_num = df_name.split('_')[-1]
    
    print(f"Memproses {df_name}...")
    
    # Mengambil DataFrame
    df = globals()[df_name]
    
    # Membuat kolom baru untuk hasil translasi Google
    df['google_translate'] = ""
    
    # Melakukan translasi untuk setiap ayat dengan tqdm untuk progress bar
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Mentranslasikan ayat"):
        try:
            # Translasi teks Arab ke Indonesia menggunakan Google Translate
            translated_text = ts.translate_text(row['arab'], from_language='ar', to_language='id', translator='google')
            df.at[idx, 'google_translate'] = translated_text
            
            # rispek rate limit
            time.sleep(1)
        except Exception as e:
            print(f"  Error pada ayat {idx+1}: {str(e)}")
            df.at[idx, 'google_translate'] = "Error: Gagal mentranslasikan"
    
    # Menyimpan hasil ke CSV
    output_path = f'./Dataset/chapter_{chapter_num}.csv'
    df.to_csv(output_path, index=False)
    print(f"Berhasil menyimpan hasil translasi ke {output_path}")

print("Proses translasi selesai untuk semua chapter")
